In [ ]:
!pip install --quiet pymupdf faiss-cpu sentence-transformers transformers
!pip install rapidfuzz
!pip install PyMuPDF


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.3 MB/s eta 0:00:00


In [ ]:
# Step 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
import os
import re
import fitz
import pickle
import numpy as np
import pandas as pd
from rapidfuzz import fuzz, process
from sentence_transformers import SentenceTransformer
import faiss




In [ ]:
DOCUMENT_STRUCTURE = {
    "PART 1: APPLICATIONS OF THE PROVISIONS OF THE CODE": [
        "TITLE & PURPOSE",
        "SCOPE & APPLICATION",
        "STRUCTURE & CHAPTER SYNOPSIS"
    ],
    "PART 2: INTERPRETATIONS": [
        "GENERAL",
        "DEFINITIONS",
        "REFERENCE STANDARDS",
        "OTHER REFERENCE DOCUMENTS"
    ],
    "PART 3: ADMINISTRATION & ENFORCEMENT": [
        "ADMINISTRATION & ENFORCEMENT",
        "PROFESSIONAL ENGAGEMENT & RESPONSIBILITIES",
        "PERMITS",
        "CONTROL & APPEALS",
        "TEMPORARY STRUCTURES & USES",
        "FEES"
    ],
    "PART 4: CLASSIFICATION OF BUILDINGS & STRUCTURES": [
        "GENERAL PRINCIPLES",
        "USE & OCCUPANCY",
        "CLASSIFICATION BY FIRE RESISTANCE",
        "MULTIPLE CLASSIFICATION",
        "HEIGHT & AREA LIMITATION",
        "DESIGN POPULATION"
    ],
    "PART 5: DEVELOPMENT PLANNING & GENERAL BUILDING REQUIREMENTS": [
        "REQUIREMENTS FOR BUILDING APPLICATION",
        "PRESENTATION OF PLANS",
        "SITE PLANS & PARTICULARS",
        "LAYOUT DRAWINGS",
        "DRAINAGE INSTALLATION DRAWINGS PARTICULARS",
        "FIRE PROTECTION PLANS",
        "GRADING, EXCAVATIONS, FILLS, NOISE & CONTROL OF DUST",
        "GENERAL PLANNING, SITING OF BUILDING & PARKING",
        "PLANNING FOR PERSONS WITH DISABILITY",
        "SUPERVISION & INSPECTION",
        "CERTIFICATION PRIOR TO OCCUPANCY",
        "VIOLATIONS, OFFENCES, STOP WORK ORDERS & FINES",
        "UNSAFE STRUCTURES",
        "FORMS"
    ],
    "PART 6: STRUCTURE": [
        "GENERAL DESIGN REQUIREMENTS",
        "LOADS, FORCES & EFFECTS",
        "GEOTECHNICAL REQUIREMENTS & FOUNDATIONS",
        "FLOORS",
        "WALLING & MASONRY",
        "PROTECTION AGAINST FALLING",
        "ROOFING STRUCTURES & RE-ROOFING",
        "CONCRETE",
        "PREFABRICATED CONCRETE",
        "DESIGN OF TIMBER STRUCTURES",
        "DESIGN OF BAMBOO STRUCTURES",
        "DESIGN OF STEEL STRUCTURES",
        "CLADDING & GLAZING",
        "SYSTEM BUILDINGS & MIXED/COMPOSITE CONSTRUCTION",
        "SPECIAL STRUCTURAL CONSTRUCTIONS"
    ],
    "PART 7: BUILDING MATERIALS": [
        "GENERAL REQUIREMENTS",
        "STRUCTURAL MATERIALS",
        "MATERIALS SPECIFICATIONS & TESTING",
        "USED, SECONDARY & ALTERNATIVE MATERIALS",
        "NON-COMPLIANCE"
    ],
    "PART 8: FIRE & SMOKE PROTECTION FEATURES": [
        "GENERAL FIRE PERFORMANCE",
        "FIRE RESISTANCE & RATING",
        "FIRE RESISTANCE TESTING",
        "FIRE WALL, BARRIERS, PARTITIONS, ENCLOSURES & OPENINGS",
        "SMOKE BARRIERS, PARTITIONS, PENETRATIONS & DUCTS",
        "FIRE PROPERTIES FOR FINISHES"
    ],
    "PART 9: FIRE DETECTION & SUPPRESSION": [
        "AUTOMATIC SPRINKLER SYSTEMS",
        "AUTOMATIC FIRE EXTINGUISHING SYSTEMS",
        "STANDPIPE SYSTEMS",
        "PORTABLE & MOBILE FIRE EXTINGUISHERS",
        "FIRE ALARMS & DETECTION SYSTEMS",
        "SMOKE CONTROL",
        "FIRE PUMPS",
        "FIRE HYDRANTS",
        "FIRE INSTALLATIONS"
    ],
    "PART 10: REQUIREMENTS FOR ACCESSIBILITY & EVACUATION": [
        "GENERAL MEANS OF EGRESS",
        "EVACUATION",
        "REQUIREMENTS FOR USE OF BUILDINGS FOR PERSONS WITH DISABILITIES"
    ],
    "PART 11: BUILDING SERVICES REQUIREMENTS": [
        "ENERGY EFFICIENCY",
        "LIGHTING & VENTILATION",
        "ELECTRICAL INSTALLATIONS",
        "AIR-CONDITIONING, HEATING & MECHANICAL VENTILATION",
        "ACOUSTICS, SOUND INSULATION & NOISE CONTROL",
        "STAIRWAYS, RAMPS & GUARDING",
        "LIFTS & ESCALATORS",
        "SECURITY SYSTEMS",
        "TELECOMMUNICATION INSTALLATIONS",
        "PLUMBING & DRAINAGE",
        "WATERBORNE SANITATION",
        "NON-WATER BORNE SANITATION",
        "SOLID WASTE MANAGEMENT"
    ],
    "PART 12: SAFEGUARDS DURING CONSTRUCTION": [
        "CONSTRUCTION SAFEGUARDS",
        "DEMOLITIONS",
        "SANITARY FACILITIES",
        "PROTECTION OF PEDESTRIANS & ADJOINING PROPERTY",
        "TEMPORARY USE OF STREETS & PUBLIC PROPERTY",
        "OTHER SAFETY MEASURES, FIRE EXTINGUISHERS & MEANS OF EGRESS"
    ],
    "PART 13: EXISTING STRUCTURES": [
        "STRUCTURAL ADDITIONS, ALTERATIONS & REPAIRS",
        "CHANGE OF OCCUPANCY",
        "FIRE ESCAPES",
        "COMPLIANCE ALTERNATIVES"
    ],
    "PART 14: INSPECTION, MAINTENANCE & DISASTER RISK MANAGEMENT": [
        "INSPECTION",
        "MAINTENANCE OF EXISTING BUILDINGS & INSTALLATIONS",
        "DISASTER RISK MANAGEMENT CONSIDERATIONS",
        "INSPECTION FORMS"
    ],
    "PART 15: SPECIAL CONDITIONS & CONSTRUCTIONS": [
        "RISK ZONING & REGIONAL CONSIDERATIONS",
        "GREEN CONSTRUCTION PRACTICES",
        "INCENTIVES FOR GREEN BUILDING",
        "LOCAL MATERIALS & CONSTRUCTION TECHNIQUES",
        "UMUDUGUDU (GROUPED SETTLEMENTS) CONSIDERATIONS",
        "INCREMENTAL BUILDING",
        "HISTORICAL BUILDINGS, MEMORIAL & BURIAL SITES"
    ]
}

#**Layer 1 Builder**

In [ ]:

# Flatten into list of allowed sections
ALLOWED_SECTIONS = [s for part, secs in DOCUMENT_STRUCTURE.items() for s in secs]

# Map section → part
SECTION_TO_PART = {sec: part for part, secs in DOCUMENT_STRUCTURE.items() for sec in secs}

# ---------------------------------------------------
# Section parsing
# ---------------------------------------------------

def normalize_section(raw_title):
    """Map noisy headings to canonical section."""
    result = process.extractOne(raw_title, ALLOWED_SECTIONS, score_cutoff=80)
    return result[0] if result else None

def find_sections(pdf_path):
    """Identify section boundaries in PDF and map them to parts."""
    doc = fitz.open(pdf_path)
    sections = []
    current_section = None
    start_page = None
    seen_sections = set()

    section_pattern = re.compile(r"^SECTION\s+\d+\s*:\s*(.+)$", re.IGNORECASE)

    for page_num, page in enumerate(doc, start=1):
        lines = page.get_text("text").split("\n")
        normalized_hits = []

        for line in lines:
            match = section_pattern.match(line.strip().upper())
            if match:
                normalized = normalize_section(match.group(1).strip())
                if normalized:
                    normalized_hits.append(normalized)

        if len(set(normalized_hits)) > 1:  # likely TOC → skip
            continue

        for normalized in normalized_hits:
            if normalized in seen_sections:
                continue
            if current_section:
                sections.append({
                    "section": current_section,
                    "part": SECTION_TO_PART.get(current_section, "UNKNOWN PART"),
                    "start_page": start_page,
                    "end_page": page_num - 1 if page_num > start_page else start_page
                })
            current_section = normalized
            start_page = page_num
            seen_sections.add(normalized)

    if current_section:
        sections.append({
            "section": current_section,
            "part": SECTION_TO_PART.get(current_section, "UNKNOWN PART"),
            "start_page": start_page,
            "end_page": doc.page_count
        })
    return sections

# ---------------------------------------------------
# Chunk builder (with metadata)
# ---------------------------------------------------

def build_layer1_keyword_chunks(sections, pdf_path, keywords, window=5, fuzz_threshold=75):
    """Extract keyword-matched chunks with ±window sentences, storing metadata with match page number."""
    doc = fitz.open(pdf_path)
    chunks = []

    for page_num, page in enumerate(doc, start=1):
        page_text = page.get_text("text")
        sentences = re.split(r'(?<=[.?!])\s+', page_text)

        # Find which section this page belongs to
        sec = next((s for s in sections if s["start_page"] <= page_num <= s["end_page"]), None)
        if not sec:
            continue  # page is outside recognized sections

        for i, sent in enumerate(sentences):
            for kw in keywords:
                if fuzz.partial_ratio(kw.lower(), sent.lower()) >= fuzz_threshold:
                    start, end = max(0, i - window), min(len(sentences), i + window + 1)
                    context = " ".join(sentences[start:end]).strip()

                    chunks.append({
                        "part": sec["part"],        # from DOCUMENT_STRUCTURE mapping
                        "section": sec["section"],  # normalized section name
                        "page": page_num,           # <-- actual page of keyword match
                        "keyword": kw,
                        "context": context
                    })
    return chunks

# ---------------------------------------------------
# FAISS index
# ---------------------------------------------------

def build_faiss_index(chunks, model_name="all-MiniLM-L6-v2"):
    model = SentenceTransformer(model_name)
    texts = [c["context"] for c in chunks]
    embeddings = model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index, embeddings, model



# ---------------------------------------------------
# Example main
# ---------------------------------------------------

if __name__ == "__main__":
    pdf_path = "/content/drive/MyDrive/Rag Wbg/Rwanda_building code_2019.pdf"
    keywords_excel_path = "/content/drive/MyDrive/Rag Wbg/layer1/keywords.xlsx"

    df = pd.read_excel(keywords_excel_path)
    keywords = df["keywords"].dropna().tolist()

    # Extract sections → map to parts
    sections = find_sections(pdf_path)

    # Build chunks with part + section + page metadata
    chunks = build_layer1_keyword_chunks(sections, pdf_path, keywords)

    # Build FAISS
    index, embeddings, model = build_faiss_index(chunks)

    # Save artifacts
    base_dir = "/content/drive/MyDrive/Rag Wbg/Fixed/layer1"
    os.makedirs(base_dir, exist_ok=True)
    faiss.write_index(index, os.path.join(base_dir, "layer1_faiss_index.bin"))
    with open(os.path.join(base_dir, "layer1_chunks.pkl"), "wb") as f:
        pickle.dump(chunks, f)
    np.save(os.path.join(base_dir, "layer1_embeddings.npy"), embeddings)



# **Layer 2 Builder**

In [ ]:

# Flatten allowed sections
ALLOWED_SECTIONS = [s for part in DOCUMENT_STRUCTURE.values() for s in part]

# Map section → part
SECTION_TO_PART = {sec: part for part, secs in DOCUMENT_STRUCTURE.items() for sec in secs}

# -------------------------------
# --- Section normalization ---
# -------------------------------
def normalize_section(raw_title):
    """Map noisy headings to canonical section names"""
    result = process.extractOne(raw_title, ALLOWED_SECTIONS, score_cutoff=80)
    if result:
        match, score, _ = result
        return match
    return None

# -------------------------------
# --- Extract sections from PDF ---
# -------------------------------
def find_sections(pdf_path):
    doc = fitz.open(pdf_path)
    sections = []
    current_section = None
    start_page = None
    seen_sections = set()

    section_pattern = re.compile(r"^SECTION\s+\d+\s*:\s*(.+)$", re.IGNORECASE)

    for page_num, page in enumerate(doc, start=1):
        lines = page.get_text("text").split("\n")
        normalized_hits = []

        for line in lines:
            line_clean = line.strip().upper()
            match = section_pattern.match(line_clean)
            if match:
                section_title = match.group(1).strip()
                normalized = normalize_section(section_title)
                if normalized:
                    normalized_hits.append(normalized)

        # Skip pages with multiple section hits (TOC)
        if len(set(normalized_hits)) > 1:
            continue

        for normalized in normalized_hits:
            if normalized in seen_sections:
                continue

            # Close previous section
            if current_section:
                sections.append({
                    "part": SECTION_TO_PART.get(current_section, "UNKNOWN"),
                    "section": current_section,
                    "start_page": start_page,
                    "end_page": page_num - 1 if page_num > start_page else start_page
                })

            current_section = normalized
            start_page = page_num
            seen_sections.add(normalized)

    # Save last section
    if current_section:
        sections.append({
            "part": SECTION_TO_PART.get(current_section, "UNKNOWN"),
            "section": current_section,
            "start_page": start_page,
            "end_page": doc.page_count
        })

    return sections

# -------------------------------
# --- Chunking utility ---
# -------------------------------
def chunk_text(text, chunk_size=500, overlap=100):
    """Split text into overlapping word chunks"""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        if end == len(words):
            break
        start = end - overlap
    return chunks

# -------------------------------
# --- Build Layer 2 embeddings ---
# -------------------------------
def build_layer2_sections(pdf_path, sections, model_name="all-MiniLM-L6-v2",
                          chunk_size=500, overlap=100):
    model = SentenceTransformer(model_name)
    layer2_chunks = []
    doc = fitz.open(pdf_path)

    for sec in sections:
        # Aggregate full text of the section
        section_text = []
        for page_num in range(sec["start_page"], sec["end_page"] + 1):
            page = doc[page_num - 1]
            section_text.append(page.get_text("text"))
        full_text = "\n".join(section_text).strip()

        if full_text:
            text_chunks = chunk_text(full_text, chunk_size=chunk_size, overlap=overlap)
            for chunk in text_chunks:
                layer2_chunks.append({
                    "part": sec["part"],
                    "section": sec["section"],
                    "page": sec["start_page"],
                    "context": chunk,
                    "confidence": 1.0
                })

    # Encode all chunks
    texts = [c["context"] for c in layer2_chunks]
    embeddings = model.encode(texts, convert_to_numpy=True, show_progress_bar=True)

    d = embeddings.shape[1]
    index = faiss.IndexFlatL2(d)
    index.add(embeddings)

    return index, embeddings, model, layer2_chunks

# -------------------------------
# --- Build and save Layer 2 ---
# -------------------------------
def build_and_save_layer2(pdf_path, base_dir):
    os.makedirs(base_dir, exist_ok=True)
    sections = find_sections(pdf_path)
    index, embeddings, model, chunks = build_layer2_sections(pdf_path, sections)

    faiss.write_index(index, os.path.join(base_dir, "layer2_faiss_index.bin"))
    with open(os.path.join(base_dir, "layer2_chunks.pkl"), "wb") as f:
        pickle.dump(chunks, f)
    np.save(os.path.join(base_dir, "layer2_embeddings.npy"), embeddings)

    print("✅ Layer 2 embeddings saved successfully!")
    return index, embeddings, model, chunks

# -------------------------------
# --- Main execution ---
# -------------------------------
if __name__ == "__main__":
    pdf_path = "/content/drive/MyDrive/Rag Wbg/Fixed/Rwanda_building code_2019.pdf"
    base_dir = "/content/drive/MyDrive/Rag Wbg/Fixed/layer2"

    index2, embeddings2, model2, layer2_chunks = build_and_save_layer2(pdf_path, base_dir)
